In [1]:
import os
import glob
import numpy as np
import pandas as pd
import networkx as nx
from scipy.linalg import pinvh, eigh
from tqdm import tqdm
from dataclasses import dataclass
from typing import List

import torch
import torch.nn as nn


### EXECUTING CELL 0


In [ ]:
# # =====================================================================
# # Global Configuration
# # =====================================================================
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Execution environment initialized on device: {DEVICE}")

# # =====================================================================
# # Module 1: Spectral Feature Extraction (Arakelov-Green Measure)
# # =====================================================================
# class SpectralTopologyExtractor:
#     def __init__(self, nx_tree: nx.Graph, nodes_coords: dict, root_node: int):
#         self.G = nx_tree.copy()
#         self.nodes_info = nodes_coords
#         self.root_node = root_node
        
#         # Kron reduction for dimensionality reduction
#         self._skeletonize_graph()
        
#         # Preserve the structural skeleton for downstream topological extraction
#         self.skeleton_tree = self.G.copy()
        
#         self.node_list = list(self.G.nodes())
#         self.node_to_idx = {n: i for i, n in enumerate(self.node_list)}
        
#     def _skeletonize_graph(self):
#         """Reduces degree-2 nodes to compact the graph structure."""
#         deg2_nodes = [n for n in self.G.nodes() if self.G.degree(n) == 2 and n != self.root_node]
#         for n in deg2_nodes:
#             neighbors = list(self.G.neighbors(n))
#             if len(neighbors) == 2:
#                 u, v = neighbors
#                 new_w = self.G[u][n]['weight'] + self.G[n][v]['weight']
#                 if self.G.has_edge(u, v):
#                     self.G[u][v]['weight'] = min(self.G[u][v]['weight'], new_w)
#                 else:
#                     self.G.add_edge(u, v, weight=new_w)
#                 self.G.remove_node(n)

#     def get_spectral_signature(self, epsilon: float = 20.0, tau: float = 5.0, top_k: int = 64) -> np.ndarray:
#         """Extracts scale-invariant spectral features using the Arakelov-Green metric."""
#         leaves = [n for n, d in self.G.degree() if d == 1 and n != self.root_node]
#         for leaf in leaves:
#             dist = np.linalg.norm(self.nodes_info[leaf] - self.nodes_info[self.root_node])
#             if dist < epsilon:
#                 self.G.add_edge(leaf, self.root_node, weight=dist)

#         bridges = set(nx.bridges(self.G))
#         n = len(self.node_list)
#         if n < 2: 
#             return np.zeros(top_k)
        
#         W = np.zeros((n, n))
#         for u, v, data in self.G.edges(data=True):
#             i, j = self.node_to_idx[u], self.node_to_idx[v]
#             is_noise = ((u, v) in bridges or (v, u) in bridges) and data['weight'] < tau
            
#             # Prevent zero-division
#             weight = max(data['weight'], 1e-5)
#             W[i, j] = W[j, i] = 1e9 if is_noise else (1.0 / weight)
            
#         D = np.diag(np.sum(W, axis=1))
        
#         # Utilize pinvh for symmetric/Hermitian matrices ensuring numerical stability
#         L_pinv = pinvh(D - W)
#         diag = np.diag(L_pinv)
#         D_AG = diag[:, np.newaxis] + diag[np.newaxis, :] - 2 * L_pinv
        
#         eigenvalues = np.sort(np.abs(eigh(D_AG, eigvals_only=True)))[::-1]
#         sig = np.zeros(top_k)
#         actual_k = min(top_k, len(eigenvalues))
#         sig[:actual_k] = eigenvalues[:actual_k]
#         return sig


# # =====================================================================
# # Module 2: Native PyTorch Batched TreeLSTM
# # =====================================================================
# class ChildSumTreeLSTMCell(nn.Module):
#     def __init__(self, x_size: int, h_size: int):
#         super().__init__()
#         self.W_iou = nn.Linear(x_size, 3 * h_size, bias=True)
#         self.U_iou = nn.Linear(h_size, 3 * h_size, bias=False)
#         self.W_f = nn.Linear(x_size, h_size, bias=True)
#         self.U_f = nn.Linear(h_size, h_size, bias=False)

#     def forward(self, x: torch.Tensor, h_children_sum: torch.Tensor, c_children_sum: torch.Tensor):
#         iou = self.W_iou(x) + self.U_iou(h_children_sum)
#         i, o, u = torch.chunk(iou, 3, dim=1)
#         i, o, u = torch.sigmoid(i), torch.sigmoid(o), torch.tanh(u)
        
#         f = torch.sigmoid(self.W_f(x) + self.U_f(h_children_sum))
        
#         c = i * u + f * c_children_sum
#         h = o * torch.tanh(c)
#         return h, c

# class NativeTreeLSTM(nn.Module):
#     def __init__(self, x_size: int, h_size: int):
#         super().__init__()
#         self.h_size = h_size
#         self.cell = ChildSumTreeLSTMCell(x_size, h_size)

#     def forward(self, x: torch.Tensor, parent_indices: torch.Tensor, levels: List[torch.Tensor], total_nodes: int):
#         # total_nodes acts as a global dummy slot for root nodes, avoiding branch checks
#         h_sum = torch.zeros(total_nodes + 1, self.h_size, device=x.device)
#         c_sum = torch.zeros(total_nodes + 1, self.h_size, device=x.device)
#         h_out = torch.zeros(total_nodes, self.h_size, device=x.device)

#         # Bottom-up propagation using breadth-first interleaved topology
#         for level_nodes in levels:
#             x_level = x[level_nodes]
#             h_sum_level = h_sum[level_nodes]
#             c_sum_level = c_sum[level_nodes]

#             h_lvl, c_lvl = self.cell(x_level, h_sum_level, c_sum_level)
#             h_out[level_nodes] = h_lvl

#             # Route outputs to parents via contiguous memory scatter operations
#             p_indices = parent_indices[level_nodes]
#             h_sum.index_add_(0, p_indices, h_lvl)
#             c_sum.index_add_(0, p_indices, c_lvl)

#         return h_out


# # =====================================================================
# # Module 3: Dataset Preparation and Graph Collation
# # =====================================================================
# @dataclass
# class GraphBatch:
#     x: torch.Tensor
#     parent_indices: torch.Tensor
#     levels: List[torch.Tensor]
#     root_indices: torch.Tensor
#     spectral_sig: torch.Tensor
#     labels: torch.Tensor
#     total_nodes: int

#     def to(self, device: torch.device):
#         return GraphBatch(
#             x=self.x.to(device),
#             parent_indices=self.parent_indices.to(device),
#             levels=[lvl.to(device) for lvl in self.levels],
#             root_indices=self.root_indices.to(device),
#             spectral_sig=self.spectral_sig.to(device),
#             labels=self.labels.to(device),
#             total_nodes=self.total_nodes
#         )

# class MorphologyDataset(torch.utils.data.Dataset):
#     def __init__(self, dataset_name: str, swc_dir: str, csv_path: str, phase: str = "train"):
#         valid_classes = {
#             "ACT-4": ["Isocortex_layer23", "Isocortex_layer4", "Isocortex_layer5", "Isocortex_layer6"],
#             "BIL-6": ["CP", "Isocortex_layer23", "Isocortex_layer4", "Isocortex_layer5", "Isocortex_layer6", "VPM"],
#             "JML-4": ["Isocortex_layer23", "Isocortex_layer5", "Isocortex_layer6", "VPM"]
#         }
#         target_classes = valid_classes[dataset_name]
#         self.label_map = {cls_name: i for i, cls_name in enumerate(target_classes)}
        
#         df = pd.read_csv(csv_path)
#         local_swc_files = glob.glob(os.path.join(swc_dir, "*.swc"))
#         local_basenames = [os.path.basename(f) for f in local_swc_files]
        
#         self.data_list = []
        
#         for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {dataset_name} ({phase})"):
#             label = row['structure_merge__acronym']
#             if isinstance(label, str): 
#                 label = label.replace("2/3", "23")
#             if label not in target_classes: 
#                 continue
                
#             fold = float(row['model__fold'])
#             if (phase == "train" and fold >= 7.0) or (phase == "val" and fold < 7.0):
#                 continue
                
#             fname_csv = str(row['swc__fname'])
#             spec_id = str(row.get('specimen__id', ''))
            
#             swc_path = next((lf for lf, l_base in zip(local_swc_files, local_basenames) 
#                              if l_base in (fname_csv, fname_csv.replace("_reg.swc", ".swc")) 
#                              or (spec_id and spec_id in l_base)), None)
#             if not swc_path: continue

#             G_tree, nodes_info, root_node = self._parse_swc(swc_path)

            
#             if len(G_tree) < 2: continue
            
#             engine = SpectralTopologyExtractor(G_tree, nodes_info, root_node)
#             spectral_sig = engine.get_spectral_signature()
#             if np.sum(spectral_sig) == 0: continue
            
#             di_g = nx.DiGraph()
#             di_g.add_nodes_from(engine.node_list)
            
#             # Explicitly construct child-to-parent directed graph
#             for u, v in nx.bfs_edges(engine.skeleton_tree, source=engine.root_node):
#                 di_g.add_edge(v, u) 
                
#             # Compute topological levels (leaves first)
#             in_degrees = dict(di_g.in_degree())
#             queue = [n for n in di_g.nodes() if in_degrees[n] == 0]
#             levels = []
#             while queue:
#                 levels.append(queue)
#                 next_q = []
#                 for n in queue:
#                     for succ in di_g.successors(n):
#                         in_degrees[succ] -= 1
#                         if in_degrees[succ] == 0: 
#                             next_q.append(succ)
#                 queue = next_q
                
#             mapping = engine.node_to_idx
#             parent_indices = np.zeros(len(engine.node_list), dtype=np.int64)
#             for child, parent in di_g.edges():
#                 parent_indices[mapping[child]] = mapping[parent]
#             parent_indices[mapping[engine.root_node]] = len(engine.node_list)
            
#             feats = np.zeros((len(engine.node_list), 5))
#             root_coords = nodes_info[engine.root_node]
            
#             # Scale-Decoupled Geometry: Local translation invariance
#             for old_id, new_id in mapping.items():
#                 feats[new_id, 0:3] = nodes_info[old_id] - root_coords
#                 feats[new_id, 3] = engine.skeleton_tree.nodes[old_id].get('type', 3)
#                 feats[new_id, 4] = engine.skeleton_tree.degree[old_id]
                
#             # Local radius normalization
#             max_radius = np.max(np.linalg.norm(feats[:, 0:3], axis=1)) + 1e-5
#             feats[:, 0:3] /= max_radius
#             feats[:, 4] /= 10.0
                
#             mapped_levels = [torch.tensor([mapping[n] for n in lvl], dtype=torch.long) for lvl in levels]
            
#             self.data_list.append({
#                 'x': torch.tensor(feats, dtype=torch.float32),
#                 'parent_indices': torch.tensor(parent_indices, dtype=torch.long),
#                 'levels': mapped_levels,
#                 'root_idx': mapping[engine.root_node],
#                 'spectral_sig': torch.tensor(spectral_sig, dtype=torch.float32),
#                 'label': self.label_map[label]
#             })

#     def _parse_swc(self, filepath: str):
#         G = nx.Graph()
#         nodes_info, edges = {}, []
#         root_node = None
#         with open(filepath, 'r') as f:
#             for line in f:
#                 if not line.strip() or line.startswith('#'): continue
#                 parts = line.split()
#                 if len(parts) < 7: continue
#                 nid, ntype = int(parts[0]), int(parts[1])
#                 x, y, z, pid = float(parts[2]), float(parts[3]), float(parts[4]), int(parts[6])
#                 nodes_info[nid] = np.array([x, y, z])
#                 G.add_node(nid, type=ntype)
#                 if pid == -1: 
#                     root_node = nid
#                 else: 
#                     edges.append((nid, pid))
                    
#         if root_node is None and len(G) > 0: 
#             root_node = list(G.nodes())[0]
            
#         for u, v in edges:
#             if u in nodes_info and v in nodes_info:
#                 G.add_edge(u, v, weight=max(np.linalg.norm(nodes_info[u]-nodes_info[v]), 1e-5))
                
#         if len(G) > 0:
#             lcc = max(nx.connected_components(G), key=len)
#             G = G.subgraph(lcc).copy()
#             if root_node not in G: 
#                 root_node = list(G.nodes())[0]
#         return G, nodes_info, root_node

#     def __len__(self): return len(self.data_list)
#     def __getitem__(self, idx): return self.data_list[idx]


# def graph_collate_fn(batch: List[dict]) -> GraphBatch:
#     total_nodes = sum(b['x'].shape[0] for b in batch)
#     x_list, batched_parents, root_indices, spectral_sigs, labels = [], [], [], [], []
    
#     offset = 0
#     offsets = []
#     for b in batch:
#         n = b['x'].shape[0]
#         x_list.append(b['x'])
        
#         p_idx = b['parent_indices'].clone()
#         mask = p_idx < n
#         p_idx[mask] += offset
        
#         # Route root pointers to the global dummy node
#         p_idx[~mask] = total_nodes 
#         batched_parents.append(p_idx)
        
#         root_indices.append(b['root_idx'] + offset)
#         spectral_sigs.append(b['spectral_sig'])
#         labels.append(b['label'])
        
#         offsets.append(offset)
#         offset += n
        
#     # Interleave nodes of depth d across the entire batch
#     max_depth = max(len(b['levels']) for b in batch)
#     batched_levels = []
#     for d in range(max_depth):
#         lvl_nodes = []
#         for i, b in enumerate(batch):
#             if d < len(b['levels']):
#                 lvl_nodes.append(b['levels'][d] + offsets[i])
#         batched_levels.append(torch.cat(lvl_nodes))
        
#     return GraphBatch(
#         x=torch.cat(x_list),
#         parent_indices=torch.cat(batched_parents),
#         levels=batched_levels,
#         root_indices=torch.tensor(root_indices, dtype=torch.long),
#         spectral_sig=torch.stack(spectral_sigs),
#         labels=torch.tensor(labels, dtype=torch.long),
#         total_nodes=total_nodes
#     )


# # =====================================================================
# # Module 4: Multimodal Fusion & Execution
# # =====================================================================
# class TopologicalTreeLSTM(nn.Module):
#     def __init__(self, num_classes: int, node_feat_dim: int = 5, hidden_dim: int = 128, spectral_dim: int = 64):
#         super().__init__()
#         self.treelstm = NativeTreeLSTM(x_size=node_feat_dim, h_size=hidden_dim)
        
#         self.graph_proj = nn.Sequential(
#             nn.Linear(hidden_dim, hidden_dim // 2),
#             nn.LayerNorm(hidden_dim // 2),
#             nn.ReLU()
#         )
        
#         self.spectral_proj = nn.Sequential(
#             nn.Linear(spectral_dim, hidden_dim // 2),
#             nn.BatchNorm1d(hidden_dim // 2),
#             nn.ReLU()
#         )
        
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(hidden_dim, num_classes)
#         )

#     def forward(self, batch: GraphBatch):
#         h_all = self.treelstm(batch.x, batch.parent_indices, batch.levels, batch.total_nodes)
#         h_root = h_all[batch.root_indices]
        
#         h_graph = self.graph_proj(h_root)
#         h_spec = self.spectral_proj(batch.spectral_sig)
        
#         h_fused = torch.cat([h_graph, h_spec], dim=-1)
#         return self.classifier(h_fused)


# def train_and_evaluate(model: nn.Module, train_loader: torch.utils.data.DataLoader, val_loader: torch.utils.data.DataLoader, epochs: int = 150, patience: int = 25) -> float:
#     optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
#     criterion = nn.CrossEntropyLoss()
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
#     best_acc = 0.0
#     epochs_no_improve = 0
    
#     for epoch in range(1, epochs + 1):
#         model.train()
#         total_loss = 0.0
        
#         for batch_data in train_loader:
#             batch = batch_data.to(DEVICE)
#             optimizer.zero_grad()
            
#             logits = model(batch)
#             loss = criterion(logits, batch.labels)
#             loss.backward()
            
#             torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#             optimizer.step()
#             total_loss += loss.item()
            
#         scheduler.step()
        
#         model.eval()
#         correct, total = 0, 0
#         with torch.no_grad():
#             for batch_data in val_loader:
#                 batch = batch_data.to(DEVICE)
#                 logits = model(batch)
#                 preds = torch.argmax(logits, dim=-1)
#                 correct += (preds == batch.labels).sum().item()
#                 total += batch.labels.size(0)
                
#         acc = correct / total
#         avg_loss = total_loss / len(train_loader)
        
#         if acc > best_acc: 
#             best_acc = acc
#             epochs_no_improve = 0
#             status = "Best Record Updated"
#         else:
#             epochs_no_improve += 1
#             status = f"Patience: {epochs_no_improve}/{patience}"
            
#         print(f"Epoch {epoch:03d}/{epochs} | Train Loss: {avg_loss:.4f} | Val Acc: {acc:.4f} | Best Acc: {best_acc:.4f} | {status}")
        
#         if epochs_no_improve >= patience:
#             print("Early stopping triggered.")
#             break
            
#     return best_acc

In [2]:
# =====================================================================
# Global Configuration
# =====================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution environment initialized on device: {DEVICE}")

# =====================================================================
# Module 1: Spectral Feature Extraction (Arakelov-Green Measure)
# =====================================================================
class SpectralTopologyExtractor:
    def __init__(self, nx_tree: nx.Graph, nodes_coords: dict, root_node: int):
        self.G = nx_tree.copy()
        self.nodes_info = nodes_coords
        self.root_node = root_node
        
        # Kron reduction for dimensionality reduction
        self._skeletonize_graph()
        
        # Preserve the structural skeleton for downstream topological extraction
        self.skeleton_tree = self.G.copy()
        
        self.node_list = list(self.G.nodes())
        self.node_to_idx = {n: i for i, n in enumerate(self.node_list)}
        
    def _skeletonize_graph(self):
        """Reduces degree-2 nodes to compact the graph structure."""
        deg2_nodes = [n for n in self.G.nodes() if self.G.degree(n) == 2 and n != self.root_node]
        for n in deg2_nodes:
            neighbors = list(self.G.neighbors(n))
            if len(neighbors) == 2:
                u, v = neighbors
                new_w = self.G[u][n]['weight'] + self.G[n][v]['weight']
                if self.G.has_edge(u, v):
                    self.G[u][v]['weight'] = min(self.G[u][v]['weight'], new_w)
                else:
                    self.G.add_edge(u, v, weight=new_w)
                self.G.remove_node(n)

    def get_spectral_signature(self, epsilon: float = 20.0, tau: float = 5.0, top_k: int = 64) -> np.ndarray:
        """Extracts scale-invariant spectral features using the Arakelov-Green metric."""
        leaves = [n for n, d in self.G.degree() if d == 1 and n != self.root_node]
        for leaf in leaves:
            dist = np.linalg.norm(self.nodes_info[leaf] - self.nodes_info[self.root_node])
            if dist < epsilon:
                self.G.add_edge(leaf, self.root_node, weight=dist)

        bridges = set(nx.bridges(self.G))
        n = len(self.node_list)
        if n < 2: 
            return np.zeros(top_k)
        
        W = np.zeros((n, n))
        for u, v, data in self.G.edges(data=True):
            i, j = self.node_to_idx[u], self.node_to_idx[v]
            is_noise = ((u, v) in bridges or (v, u) in bridges) and data['weight'] < tau
            
            # Prevent zero-division
            weight = max(data['weight'], 1e-5)
            W[i, j] = W[j, i] = 1e9 if is_noise else (1.0 / weight)
            
        D = np.diag(np.sum(W, axis=1))
        
        # Utilize pinvh for symmetric/Hermitian matrices ensuring numerical stability
        L_pinv = pinvh(D - W)
        diag = np.diag(L_pinv)
        D_AG = diag[:, np.newaxis] + diag[np.newaxis, :] - 2 * L_pinv
        
        eigenvalues = np.sort(np.abs(eigh(D_AG, eigvals_only=True)))[::-1]
        sig = np.zeros(top_k)
        actual_k = min(top_k, len(eigenvalues))
        sig[:actual_k] = eigenvalues[:actual_k]
        return sig


# =====================================================================
# Module 2: Native PyTorch Batched TreeLSTM
# =====================================================================
class ChildSumTreeLSTMCell(nn.Module):
    def __init__(self, x_size: int, h_size: int):
        super().__init__()
        self.W_iou = nn.Linear(x_size, 3 * h_size, bias=True)
        self.U_iou = nn.Linear(h_size, 3 * h_size, bias=False)
        self.W_f = nn.Linear(x_size, h_size, bias=True)
        self.U_f = nn.Linear(h_size, h_size, bias=False)

    def forward(self, x: torch.Tensor, h_children_sum: torch.Tensor, c_children_weighted: torch.Tensor):
        iou = self.W_iou(x) + self.U_iou(h_children_sum)
        i, o, u = torch.chunk(iou, 3, dim=1)
        i, o, u = torch.sigmoid(i), torch.sigmoid(o), torch.tanh(u)

        c = i * u + c_children_weighted
        h = o * torch.tanh(c)
        return h, c

    def forget_contribution(
        self,
        parent_x: torch.Tensor,
        child_h: torch.Tensor,
        child_c: torch.Tensor,
    ) -> torch.Tensor:
        forget_gate = torch.sigmoid(self.W_f(parent_x) + self.U_f(child_h))
        return forget_gate * child_c

class NativeTreeLSTM(nn.Module):
    def __init__(self, x_size: int, h_size: int):
        super().__init__()
        self.h_size = h_size
        self.cell = ChildSumTreeLSTMCell(x_size, h_size)

    def forward(self, x: torch.Tensor, parent_indices: torch.Tensor, levels: List[torch.Tensor], total_nodes: int):
        # total_nodes acts as a global dummy slot for root nodes, avoiding branch checks
        h_sum = torch.zeros(total_nodes + 1, self.h_size, device=x.device)
        c_weighted = torch.zeros(total_nodes + 1, self.h_size, device=x.device)
        h_out = torch.zeros(total_nodes, self.h_size, device=x.device)

        # Bottom-up propagation using breadth-first interleaved topology
        for level_nodes in levels:
            x_level = x[level_nodes]
            h_sum_level = h_sum[level_nodes]
            c_weighted_level = c_weighted[level_nodes]

            h_lvl, c_lvl = self.cell(x_level, h_sum_level, c_weighted_level)
            h_out[level_nodes] = h_lvl

            # Route each child through its own forget gate before parent aggregation.
            p_indices = parent_indices[level_nodes]
            h_sum.index_add_(0, p_indices, h_lvl)

            valid_parent = p_indices < total_nodes
            valid_parent_indices = p_indices[valid_parent]
            forget_c = self.cell.forget_contribution(
                x[valid_parent_indices],
                h_lvl[valid_parent],
                c_lvl[valid_parent],
            )
            c_weighted.index_add_(0, valid_parent_indices, forget_c)

        return h_out


# =====================================================================
# Module 3: Dataset Preparation and Graph Collation
# =====================================================================
@dataclass
class GraphBatch:
    x: torch.Tensor
    parent_indices: torch.Tensor
    levels: List[torch.Tensor]
    batch_indices: torch.Tensor
    root_indices: torch.Tensor
    spectral_sig: torch.Tensor
    labels: torch.Tensor
    total_nodes: int

    def to(self, device: torch.device):
        return GraphBatch(
            x=self.x.to(device),
            parent_indices=self.parent_indices.to(device),
            levels=[lvl.to(device) for lvl in self.levels],
            batch_indices=self.batch_indices.to(device),
            root_indices=self.root_indices.to(device),
            spectral_sig=self.spectral_sig.to(device),
            labels=self.labels.to(device),
            total_nodes=self.total_nodes
        )

class MorphologyDataset(torch.utils.data.Dataset):
    def __init__(self, dataset_name: str, swc_dir: str, csv_path: str, phase: str = "train"):
        valid_classes = {
            "ACT-4": ["Isocortex_layer23", "Isocortex_layer4", "Isocortex_layer5", "Isocortex_layer6"],
            "BIL-6": ["CP", "Isocortex_layer23", "Isocortex_layer4", "Isocortex_layer5", "Isocortex_layer6", "VPM"],
            "JML-4": ["Isocortex_layer23", "Isocortex_layer5", "Isocortex_layer6", "VPM"]
        }
        target_classes = valid_classes[dataset_name]
        self.label_map = {cls_name: i for i, cls_name in enumerate(target_classes)}
        
        df = pd.read_csv(csv_path)


    
        # Set here
        local_swc_files = set(glob.glob(os.path.join(swc_dir, "*.swc")))
        
        self.data_list = []

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {dataset_name} ({phase})"):
            label = row['structure_merge__acronym']
            if isinstance(label, str): 
                label = label.replace("2/3", "23")
            if label not in target_classes: 
                continue
                
            fold = float(row['model__fold'])
            
            # 
            if phase == "train" and fold >= 8.0: 
                continue
            if phase == "val" and fold < 8.0: 
                continue
                
            fname_csv = str(row['swc__fname'])
            exact_swc_path = os.path.join(swc_dir, fname_csv)
            
            if exact_swc_path not in local_swc_files:
                continue
                
            swc_path = exact_swc_path
            

            
            # 
            G_tree, nodes_info, root_node = self._parse_swc(swc_path)
            

        
            
            if len(G_tree) < 2: continue
            
            engine = SpectralTopologyExtractor(G_tree, nodes_info, root_node)
            spectral_sig = engine.get_spectral_signature()
            if np.sum(spectral_sig) == 0: continue
            
            di_g = nx.DiGraph()
            di_g.add_nodes_from(engine.node_list)
            
            # Explicitly construct child-to-parent directed graph
            for u, v in nx.bfs_edges(engine.skeleton_tree, source=engine.root_node):
                di_g.add_edge(v, u) 
                
            # Compute topological levels (leaves first)
            in_degrees = dict(di_g.in_degree())
            queue = [n for n in di_g.nodes() if in_degrees[n] == 0]
            levels = []
            while queue:
                levels.append(queue)
                next_q = []
                for n in queue:
                    for succ in di_g.successors(n):
                        in_degrees[succ] -= 1
                        if in_degrees[succ] == 0: 
                            next_q.append(succ)
                queue = next_q
                
            mapping = engine.node_to_idx
            parent_indices = np.zeros(len(engine.node_list), dtype=np.int64)
            for child, parent in di_g.edges():
                parent_indices[mapping[child]] = mapping[parent]
            parent_indices[mapping[engine.root_node]] = len(engine.node_list)
            
            feats = np.zeros((len(engine.node_list), 5))
            root_coords = nodes_info[engine.root_node]
            
            # Scale-Decoupled Geometry: Local translation invariance
            for old_id, new_id in mapping.items():
                feats[new_id, 0:3] = nodes_info[old_id] - root_coords
                feats[new_id, 3] = engine.skeleton_tree.nodes[old_id].get('type', 3)
                feats[new_id, 4] = engine.skeleton_tree.degree[old_id]
                
            # Local radius normalization
            max_radius = np.max(np.linalg.norm(feats[:, 0:3], axis=1)) + 1e-5
            feats[:, 0:3] /= max_radius
            feats[:, 4] /= 10.0
                
            mapped_levels = [torch.tensor([mapping[n] for n in lvl], dtype=torch.long) for lvl in levels]
            
            self.data_list.append({
                'x': torch.tensor(feats, dtype=torch.float32),
                'parent_indices': torch.tensor(parent_indices, dtype=torch.long),
                'levels': mapped_levels,
                'root_idx': mapping[engine.root_node],
                'spectral_sig': torch.tensor(spectral_sig, dtype=torch.float32),
                'label': self.label_map[label]
            })

    def _parse_swc(self, filepath: str):
        G = nx.Graph()
        nodes_info, edges = {}, []
        root_node = None
        with open(filepath, 'r') as f:
            for line in f:
                if not line.strip() or line.startswith('#'): continue
                parts = line.split()
                if len(parts) < 7: continue
                nid, ntype = int(parts[0]), int(parts[1])
                x, y, z, pid = float(parts[2]), float(parts[3]), float(parts[4]), int(parts[6])
                nodes_info[nid] = np.array([x, y, z])
                G.add_node(nid, type=ntype)
                if pid == -1: 
                    root_node = nid
                else: 
                    edges.append((nid, pid))
                    
        if root_node is None and len(G) > 0: 
            root_node = list(G.nodes())[0]
            
        for u, v in edges:
            if u in nodes_info and v in nodes_info:
                G.add_edge(u, v, weight=max(np.linalg.norm(nodes_info[u]-nodes_info[v]), 1e-5))
                
        if len(G) > 0:
            lcc = max(nx.connected_components(G), key=len)
            G = G.subgraph(lcc).copy()
            if root_node not in G: 
                root_node = list(G.nodes())[0]
        return G, nodes_info, root_node

    def __len__(self): return len(self.data_list)
    def __getitem__(self, idx): return self.data_list[idx]


def graph_collate_fn(batch: List[dict]) -> GraphBatch:
    total_nodes = sum(b['x'].shape[0] for b in batch)
    x_list, batched_parents, batch_indices, root_indices, spectral_sigs, labels = [], [], [], [], [], []

    offset = 0
    offsets = []
    for graph_idx, b in enumerate(batch):
        n = b['x'].shape[0]
        x_list.append(b['x'])
        batch_indices.append(torch.full((n,), graph_idx, dtype=torch.long))
        
        p_idx = b['parent_indices'].clone()
        mask = p_idx < n
        p_idx[mask] += offset
        
        # Route root pointers to the global dummy node
        p_idx[~mask] = total_nodes 
        batched_parents.append(p_idx)
        
        root_indices.append(b['root_idx'] + offset)
        spectral_sigs.append(b['spectral_sig'])
        labels.append(b['label'])
        
        offsets.append(offset)
        offset += n
        
    # Interleave nodes of depth d across the entire batch
    max_depth = max(len(b['levels']) for b in batch)
    batched_levels = []
    for d in range(max_depth):
        lvl_nodes = []
        for i, b in enumerate(batch):
            if d < len(b['levels']):
                lvl_nodes.append(b['levels'][d] + offsets[i])
        batched_levels.append(torch.cat(lvl_nodes))
        
    return GraphBatch(
        x=torch.cat(x_list),
        parent_indices=torch.cat(batched_parents),
        levels=batched_levels,
        batch_indices=torch.cat(batch_indices),
        root_indices=torch.tensor(root_indices, dtype=torch.long),
        spectral_sig=torch.stack(spectral_sigs),
        labels=torch.tensor(labels, dtype=torch.long),
        total_nodes=total_nodes
    )


# =====================================================================
# Module 4: Multimodal Fusion & Execution
# =====================================================================
class TopologicalTreeLSTM(nn.Module):
    def __init__(self, num_classes: int, node_feat_dim: int = 5, hidden_dim: int = 128, spectral_dim: int = 64):
        super().__init__()
        self.treelstm = NativeTreeLSTM(x_size=node_feat_dim, h_size=hidden_dim)
        
        self.graph_proj = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU()
        )
        
        self.spectral_proj = nn.Sequential(
            nn.Linear(spectral_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU()
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, batch: GraphBatch):
        h_all = self.treelstm(batch.x, batch.parent_indices, batch.levels, batch.total_nodes)
        h_root = h_all[batch.root_indices]

        num_graphs = batch.root_indices.numel()
        h_sum = torch.zeros(num_graphs, h_all.shape[1], device=h_all.device)
        h_sum.index_add_(0, batch.batch_indices, h_all)
        node_counts = torch.bincount(batch.batch_indices, minlength=num_graphs).unsqueeze(1)
        h_mean = h_sum / node_counts.clamp_min(1)

        h_max = torch.full_like(h_sum, -torch.inf)
        expanded_batch_indices = batch.batch_indices.unsqueeze(1).expand_as(h_all)
        h_max.scatter_reduce_(0, expanded_batch_indices, h_all, reduce="amax", include_self=True)

        h_graph = self.graph_proj(torch.cat([h_root, h_mean, h_max], dim=-1))
        h_spec = self.spectral_proj(batch.spectral_sig)
        
        h_fused = torch.cat([h_graph, h_spec], dim=-1)
        return self.classifier(h_fused)


def train_and_evaluate(model: nn.Module, train_loader: torch.utils.data.DataLoader, val_loader: torch.utils.data.DataLoader, epochs: int = 150, patience: int = 25) -> float:
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    best_acc = 0.0
    epochs_no_improve = 0
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        
        for batch_data in train_loader:
            batch = batch_data.to(DEVICE)
            optimizer.zero_grad()
            
            logits = model(batch)
            loss = criterion(logits, batch.labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step()
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch_data in val_loader:
                batch = batch_data.to(DEVICE)
                logits = model(batch)
                preds = torch.argmax(logits, dim=-1)
                correct += (preds == batch.labels).sum().item()
                total += batch.labels.size(0)
                
        acc = correct / total
        avg_loss = total_loss / len(train_loader)
        
        if acc > best_acc: 
            best_acc = acc
            epochs_no_improve = 0
            status = "Best Record Updated"
        else:
            epochs_no_improve += 1
            status = f"Patience: {epochs_no_improve}/{patience}"
            
        print(f"Epoch {epoch:03d}/{epochs} | Train Loss: {avg_loss:.4f} | Val Acc: {acc:.4f} | Best Acc: {best_acc:.4f} | {status}")
        
        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break
            
    return best_acc


### EXECUTING CELL 3
Execution environment initialized on device: cuda


In [ ]:
# def main():
#     datasets_config = [
#         ("ACT-4", "./data/raw/allen_cell_type/swc", "./data/info/ACT_info_swc_10folds.csv", 4),
#         ("JML-4", "./data/raw/janelia_mouselight/swc", "./data/info/JML_info_swc_10folds.csv", 4),
#         ("BIL-6", "./data/raw/bil/swc", "./data/info/BIL_info_swc_10folds.csv", 6)
#     ]
    
#     batch_size = 128
    
#     for ds_name, swc_dir, csv_path, num_classes in datasets_config:
#         print("\n" + "="*60)
#         print(f"Dataset Evaluation Initialized: {ds_name}")
        
#         train_ds = MorphologyDataset(ds_name, swc_dir, csv_path, phase="train")
#         val_ds = MorphologyDataset(ds_name, swc_dir, csv_path, phase="val")
        
#         if len(train_ds) == 0 or len(val_ds) == 0: 
#             print(f"Dataset {ds_name} files unlocated or empty. Skipping evaluation.")
#             continue
            
#         train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, collate_fn=graph_collate_fn, shuffle=True)
#         val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, collate_fn=graph_collate_fn, shuffle=False)
        
#         model = TopologicalTreeLSTM(num_classes=num_classes).to(DEVICE)
#         best_acc = train_and_evaluate(model, train_loader, val_loader, epochs=150, patience=25)
        
#         print(f"Final Accuracy for {ds_name}: {best_acc * 100:.2f}%")

In [ ]:
# if __name__ == "__main__":
#     main()

In [ ]:
# 8:2 BIL6 BUG fixed

In [3]:
import random

# =====================================================================
# Seed Initialization Utility
# =====================================================================
def set_seed(seed: int):
    """
    Ensures absolute reproducibility across DataLoader shuffling batches 
    and neural network parameter initialization.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Enforce deterministic backend for convolutions and matrix operations
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =====================================================================
# Main Execution (Multi-Seed Statistical Evaluation)
# =====================================================================
def main():
    print(f"Environment initialized. Launching TopologicalTreeLSTM on {DEVICE}.")
    print("Note: Multi-seed robust evaluation activated.\n")
    
    datasets_config = [
        ("ACT-4", "./data/raw/allen_cell_type/swc", "./data/info/ACT_info_swc_10folds.csv", 4),
        ("JML-4", "./data/raw/janelia_mouselight/swc", "./data/info/JML_info_swc_10folds.csv", 4),
        ("BIL-6", "./data/raw/bil/swc", "./data/info/BIL_info_swc_10folds.csv", 6)
    ]
    
    seeds = [42, 114514, 1919810, 666, 1453]
    batch_size = 128
    
    for ds_name, swc_dir, csv_path, num_classes in datasets_config:
        print("\n" + "="*70)
        print(f"Dataset Evaluation Initialized: {ds_name}")
        
        # Engineering Optimization: 
        # Since TreeLSTM morphological extraction is purely deterministic (no random sampling),
        # Dataset instantiation is moved outside the seed loop to avoid massive redundant CPU computations.
        train_ds = MorphologyDataset(ds_name, swc_dir, csv_path, phase="train")
        val_ds = MorphologyDataset(ds_name, swc_dir, csv_path, phase="val")
        
        if len(train_ds) == 0 or len(val_ds) == 0: 
            print(f"Dataset {ds_name} files unlocated or empty. Skipping evaluation.")
            continue
            
        seed_results = []
        
        for seed in seeds:
            print(f"\n--- Initiating Run with Seed: {seed} ---")
            set_seed(seed)
            
            # DataLoader is instantiated inside the loop to bind the specific seed to the shuffle generator
            train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, collate_fn=graph_collate_fn, shuffle=True)
            val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, collate_fn=graph_collate_fn, shuffle=False)
            
            # Model instantiation strictly inside the loop ensures distinct weight initializations per seed
            model = TopologicalTreeLSTM(num_classes=num_classes).to(DEVICE)
            best_acc = train_and_evaluate(model, train_loader, val_loader, epochs=150, patience=25)
            
            seed_results.append(best_acc)
            print(f"Run Completed [Seed {seed}]. Best Accuracy: {best_acc * 100:.2f}%")
            
        # Aggregate and report statistical robustness
        if seed_results:
            results_array = np.array(seed_results) * 100
            mean_acc = np.mean(results_array)
            std_acc = np.std(results_array)
            
            print("\n" + "*"*70)
            print(f"Final Statistical Evaluation for {ds_name} ({len(seeds)} Seeds)")
            print(f"Mean ± Std: {mean_acc:.2f}% ± {std_acc:.2f}%")
            print("*"*70)

if __name__ == "__main__":
    main()


### EXECUTING CELL 11
Environment initialized. Launching TopologicalTreeLSTM on cuda.
Note: Multi-seed robust evaluation activated.


Dataset Evaluation Initialized: ACT-4

--- Initiating Run with Seed: 42 ---
Epoch 001/150 | Train Loss: 1.3474 | Val Acc: 0.3158 | Best Acc: 0.3158 | Best Record Updated
Epoch 002/150 | Train Loss: 1.3072 | Val Acc: 0.3158 | Best Acc: 0.3158 | Patience: 1/25
Epoch 003/150 | Train Loss: 1.3064 | Val Acc: 0.3053 | Best Acc: 0.3158 | Patience: 2/25
Epoch 004/150 | Train Loss: 1.3220 | Val Acc: 0.3158 | Best Acc: 0.3158 | Patience: 3/25
Epoch 005/150 | Train Loss: 1.2671 | Val Acc: 0.3053 | Best Acc: 0.3158 | Patience: 4/25
Epoch 006/150 | Train Loss: 1.2707 | Val Acc: 0.3158 | Best Acc: 0.3158 | Patience: 5/25
Epoch 007/150 | Train Loss: 1.2404 | Val Acc: 0.3263 | Best Acc: 0.3263 | Best Record Updated
Epoch 008/150 | Train Loss: 1.3035 | Val Acc: 0.3474 | Best Acc: 0.3474 | Best Record Updated
Epoch 009/150 | Train Loss: 1.2097 | Val Acc: 0.3684 | Best Ac

In [ ]:
# 7：3 BIL BUG

In [ ]:
import random

# =====================================================================
# Seed Initialization Utility
# =====================================================================
def set_seed(seed: int):
    """
    Ensures absolute reproducibility across DataLoader shuffling batches 
    and neural network parameter initialization.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Enforce deterministic backend for convolutions and matrix operations
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =====================================================================
# Main Execution (Multi-Seed Statistical Evaluation)
# =====================================================================
def main():
    print(f"Environment initialized. Launching TopologicalTreeLSTM on {DEVICE}.")
    print("Note: Multi-seed robust evaluation activated.\n")
    
    datasets_config = [
        ("ACT-4", "./data/raw/allen_cell_type/swc", "./data/info/ACT_info_swc_10folds.csv", 4),
        ("JML-4", "./data/raw/janelia_mouselight/swc", "./data/info/JML_info_swc_10folds.csv", 4),
        ("BIL-6", "./data/raw/bil/swc", "./data/info/BIL_info_swc_10folds.csv", 6)
    ]
    
    seeds = [42, 114514, 1919810, 666, 1453]
    batch_size = 128
    
    for ds_name, swc_dir, csv_path, num_classes in datasets_config:
        print("\n" + "="*70)
        print(f"Dataset Evaluation Initialized: {ds_name}")
        
        # Engineering Optimization: 
        # Since TreeLSTM morphological extraction is purely deterministic (no random sampling),
        # Dataset instantiation is moved outside the seed loop to avoid massive redundant CPU computations.
        train_ds = MorphologyDataset(ds_name, swc_dir, csv_path, phase="train")
        val_ds = MorphologyDataset(ds_name, swc_dir, csv_path, phase="val")
        
        if len(train_ds) == 0 or len(val_ds) == 0: 
            print(f"Dataset {ds_name} files unlocated or empty. Skipping evaluation.")
            continue
            
        seed_results = []
        
        for seed in seeds:
            print(f"\n--- Initiating Run with Seed: {seed} ---")
            set_seed(seed)
            
            # DataLoader is instantiated inside the loop to bind the specific seed to the shuffle generator
            train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, collate_fn=graph_collate_fn, shuffle=True)
            val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, collate_fn=graph_collate_fn, shuffle=False)
            
            # Model instantiation strictly inside the loop ensures distinct weight initializations per seed
            model = TopologicalTreeLSTM(num_classes=num_classes).to(DEVICE)
            best_acc = train_and_evaluate(model, train_loader, val_loader, epochs=150, patience=25)
            
            seed_results.append(best_acc)
            print(f"Run Completed [Seed {seed}]. Best Accuracy: {best_acc * 100:.2f}%")
            
        # Aggregate and report statistical robustness
        if seed_results:
            results_array = np.array(seed_results) * 100
            mean_acc = np.mean(results_array)
            std_acc = np.std(results_array)
            
            print("\n" + "*"*70)
            print(f"Final Statistical Evaluation for {ds_name} ({len(seeds)} Seeds)")
            print(f"Mean ± Std: {mean_acc:.2f}% ± {std_acc:.2f}%")
            print("*"*70)

if __name__ == "__main__":
    main()